# IMS Tutorial (Repository-based)

This notebook demonstrates how to **register** and **use** the IMS toolkit as a repository-backed datasource in Hera.
It mirrors the previous IMS tutorial but adapts it to the new requirement that every datasource is tied to a **repository** (by name) and retrieved via *(repository, datasourceName)*.

## What you will do
1. Set up paths and make sure Python can import IMS and Argos (pyargos).
2. Register the IMS toolkit into a project’s repository via the CLI.
3. Verify the registration and inspect the stored document.
4. Dynamically import and instantiate the IMS class using the stored classpath and parameters.
5. (Optional) Run example operations (download/update) once your token is configured.

## Prerequisites
- Local folder `~/hera-ims` with:
  - `code/` containing `IMS_experiment.py` and a compatible `presentation.py` (or equivalent).
  - `data/` (will store parquet files).
  - `token.json` with your IMS API token (used by the class).
- Local Argos (pyargos) checkout at `~/pyargos-master` (adjust paths below if different).
- Hera environment installed/activated.


## 1) Paths & Environment Setup
We define a project name and the IMS directory structure. We also ensure the search path (`sys.path`) includes our virtualenv's site-packages, the IMS code folder, its root, and the pyargos root.
If your paths differ, edit the variables below.

In [10]:
import os, sys
PROJECT  = "UnitTestProject"
IMS_ROOT = os.path.expanduser("~/hera-ims")
IMS_CODE = os.path.join(IMS_ROOT, "code")
IMS_DATA = os.path.join(IMS_ROOT, "data")

# Make sure folders exist and that 'code' is a package:
os.makedirs(IMS_CODE, exist_ok=True)
os.makedirs(IMS_DATA, exist_ok=True)
open(os.path.join(IMS_CODE, "__init__.py"), "a").close()

# Common locations – adjust if needed:
VENV_SP = os.path.expanduser("~/hera/heraenv/lib/python3.9/site-packages")
PYARGOS = os.path.expanduser("~/pyargos-master")

# Put search paths at the front
for p in (VENV_SP, IMS_CODE, IMS_ROOT, PYARGOS):
    if p and p not in sys.path:
        sys.path.insert(0, p)

print("✔ Environment ready")
print("  Project:", PROJECT)
print("  IMS code dir:", IMS_CODE)
print("  IMS data dir:", IMS_DATA)
print("  pyargos root:", PYARGOS)


✔ Environment ready
  Project: UnitTestProject
  IMS code dir: /home/ilay/hera-ims/code
  IMS data dir: /home/ilay/hera-ims/data
  pyargos root: /home/ilay/pyargos-master


## 2) Register the IMS datasource (CLI)
We use Hera's repository CLI to register a **ToolkitDataSource** named `IMS` into the repository `IMS` within the project `UnitTestProject`.

**Key arguments**:
- `--project`: the Hera project name.
- `--name`: the datasource name (unique within the repository).
- `--repository`: the repository name that holds the datasource.
- `--classpath`: full Python path to the toolkit class (here we use `code.IMS_experiment.IMS_experiment`).
- `--params`: JSON with constructor parameters for the class (project name, experiment path, data path).
- `--overwrite`: replace existing entry if it already exists.

If you keep your IMS class as a top-level module (e.g., `IMS_experiment.IMS_experiment`), you can switch `--classpath` accordingly **as long as** it is importable at runtime.

In [11]:
import os, sys, json, subprocess

# Build params JSON for the toolkit constructor
params = json.dumps({
    "projectName": PROJECT,
    "pathToExperiment": IMS_ROOT,
    "filesDirectory": IMS_DATA,
})

# Ensure CLI can import everything (venv first)
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    os.path.expanduser("~/hera/heraenv/lib/python3.9/site-packages"),
    IMS_CODE, IMS_ROOT, os.path.expanduser("~/pyargos-master"),
    env.get("PYTHONPATH","")
])

# Register the datasource into repository 'IMS'
subprocess.run([
    sys.executable, "-m", "hera.utils.data.cli_toolkit_repository", "register-datasource",
    "--project", PROJECT,
    "--name", "IMS",
    "--repository", "IMS",
    "--classpath", "code.IMS_experiment.IMS_experiment",
    "--params", params,
    "--version", "0.0.1",
    "--overwrite",
], check=True, env=env)

print("✔ Registered 'IMS' datasource into repository 'IMS'")


/home/ilay/hera/heraenv/lib/python3.9/site-packages/xarray/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


qgl is not installed
Thingsboard interface not installed. Use pip install tb_rest_client.
Registered datasource:
  project     : UnitTestProject
  repository  : IMS
  name        : IMS
  classpath   : code.IMS_experiment.IMS_experiment
  resource    : /home/ilay/hera-ims
  parameters  : {'projectName': 'UnitTestProject', 'pathToExperiment': '/home/ilay/hera-ims', 'filesDirectory': '/home/ilay/hera-ims/data'}
  version     : [0, 0, 1]
✔ Registered 'IMS' datasource into repository 'IMS'


## 3) Verify Registration
Use the CLI to print the toolkits table and confirm that `IMS` appears under **external** / **measurements**.

In [12]:
import sys, subprocess
subprocess.run([
    sys.executable, "-m", "hera.utils.data.cli_toolkit_repository",
    "print", "--project", PROJECT
], check=True)


/home/ilay/hera/heraenv/lib/python3.9/site-packages/xarray/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


| toolkit               | cls                                                                | source   | type         | description                               |
|-----------------------|--------------------------------------------------------------------|----------|--------------|-------------------------------------------|
| GIS_Buildings         | hera.measurements.GIS.vector.buildings.toolkit.BuildingsToolkit    | internal | measurements |                                           |
| GIS_Tiles             | hera.measurements.GIS.raster.tiles.TilesToolkit                    | internal | measurements |                                           |
| GIS_Vector_Topography | hera.measurements.GIS.vector.topography.TopographyToolkit          | internal | measurements |                                           |
| GIS_Raster_Topography | hera.measurements.GIS.raster.topography.TopographyToolkit          | internal | measurements |                                           |
| GIS_Demo

CompletedProcess(args=['/home/ilay/hera/heraenv/bin/python', '-m', 'hera.utils.data.cli_toolkit_repository', 'print', '--project', 'UnitTestProject'], returncode=0)

## 4) Retrieve the Datasource Document (Repository + Name)
We now fetch the newly created `ToolkitDataSource` document via the **Project** API, filtering by `type='ToolkitDataSource'`, `repository='IMS'` and `datasourceName='IMS'`.
We then print its **classpath**, **resource**, **version**, and **parameters** for inspection.

In [13]:
from hera.datalayer.project import Project

proj = Project(projectName=PROJECT)
docs = proj.getMeasurementsDocuments(
    type="ToolkitDataSource",
    repository="IMS",
    datasourceName="IMS",
)
assert docs, "Datasource 'IMS' in repository 'IMS' not found."
doc = docs[0]

print("Document located ✅")
print("  repository :", doc.desc.get("repository"))
print("  name       :", doc.desc.get("datasourceName"))
print("  classpath  :", doc.desc.get("classpath"))
print("  resource   :", doc.resource)
print("  version    :", doc.desc.get("version"))
print("  parameters :", doc.desc.get("parameters"))


Document located ✅
  repository : IMS
  name       : IMS
  classpath  : code.IMS_experiment.IMS_experiment
  resource   : /home/ilay/hera-ims
  version    : [0, 0, 1]
  parameters : {'projectName': 'UnitTestProject', 'pathToExperiment': '/home/ilay/hera-ims', 'filesDirectory': '/home/ilay/hera-ims/data'}


## 5) Import and Instantiate the IMS Class
Using the stored `classpath` and `parameters`, we dynamically import the class and construct an instance. This validates that your environment (paths + dependencies) is consistent.

> **Tip**: If import fails (e.g., cannot find `presentation`), ensure your `PYTHONPATH` includes both the IMS code folder **and** the IMS root (if you import intra-package modules).

In [15]:
# Robust retrieval of the ToolkitDataSource document (Cell 5 replacement)

import os, sys, json
from pprint import pprint

# --- Make sure runtime paths are in place INSIDE this kernel ---
VENV_SP   = os.path.expanduser("~/hera/heraenv/lib/python3.9/site-packages")
IMS_ROOT  = os.path.expanduser("~/hera-ims")
IMS_CODE  = os.path.join(IMS_ROOT, "code")
PYARGOS   = os.path.expanduser("~/pyargos-master")

os.environ["PYTHONNOUSERSITE"] = "1"
for p in (VENV_SP, IMS_CODE, IMS_ROOT, PYARGOS):
    if p and p not in sys.path:
        sys.path.insert(0, p)

# Optional: sanity for typing_extensions.Self
try:
    import typing_extensions as _te
    _ = getattr(_te, "Self")
    print("✅ typing_extensions.Self OK from:", _te.__file__)
except Exception as e:
    print("⚠️ typing_extensions.Self missing or wrong package on path:", e)

# --- Fetch document ---
try:
    from hera.datalayer.project import Project
except Exception as e:
    raise RuntimeError("Failed to import hera.datalayer.project. Check sys.path ordering and venv site-packages first.") from e

# PROJECT אמור להיות מוגדר בתאים הקודמים; אם לא, אפשר להגדיר כאן:
PROJECT = globals().get("PROJECT", "UnitTestProject")

proj = Project(projectName=PROJECT)
docs = proj.getMeasurementsDocuments(
    type="ToolkitDataSource",
    repository="IMS",        # <— must match the repository you used in the CLI cell
    datasourceName="IMS",    # <— must match the datasource name
)

if not docs:
    raise RuntimeError(
        "ToolkitDataSource not found.\n"
        "• Make sure you ran the CLI registration cell successfully in THIS kernel.\n"
        "• Confirm repository='IMS' and datasourceName='IMS'.\n"
        "• If you changed classpath to top-level, ensure sys.path includes both IMS_ROOT and IMS_CODE."
    )

doc = docs[0]

print("✅ Document located")
print("  repository :", doc.desc.get("repository"))
print("  name       :", doc.desc.get("datasourceName"))
print("  classpath  :", doc.desc.get("classpath"))
print("  resource   :", doc.resource)
print("  version    :", doc.desc.get("version"))
print("  parameters :")
pprint(doc.desc.get("parameters"))


✅ typing_extensions.Self OK from: /home/ilay/hera/heraenv/lib/python3.9/site-packages/typing_extensions.py
✅ Document located
  repository : IMS
  name       : IMS
  classpath  : code.IMS_experiment.IMS_experiment
  resource   : /home/ilay/hera-ims
  version    : [0, 0, 1]
  parameters :
{'filesDirectory': '/home/ilay/hera-ims/data',
 'pathToExperiment': '/home/ilay/hera-ims',
 'projectName': 'UnitTestProject'}


## 6) Optional Usage Examples
Uncomment the calls below after you add a valid IMS token in `~/hera-ims/token.json` and ensure network access. These calls may take time and write parquet files under the IMS `data/` folder.

**Note**: pick a station name that exists in your IMS metadata. The class typically fetches metadata first and filters by name.


In [18]:
# Cell 6 — Safe usage: ensure `obj` exists, then (optionally) run examples

import os, sys, importlib, json

# --- Paths (adjust if your setup differs) ---
VENV_SP  = os.path.expanduser("~/hera/heraenv/lib/python3.9/site-packages")
IMS_ROOT = os.path.expanduser("~/hera-ims")       # folder that contains the 'code/' dir
IMS_CODE = os.path.join(IMS_ROOT, "code")
PYARGOS  = os.path.expanduser("~/pyargos-master")

os.environ["PYTHONNOUSERSITE"] = "1"

# Ensure 'code' is our package: IMS_ROOT must be on sys.path and code/ must be a package
os.makedirs(IMS_CODE, exist_ok=True)
open(os.path.join(IMS_CODE, "__init__.py"), "a").close()

# Put IMS_ROOT first so 'import code.XXX' resolves to your IMS package, not stdlib's 'code' module
for p in (IMS_ROOT, VENV_SP, PYARGOS):
    if p and p not in sys.path:
        sys.path.insert(0, p)

# If stdlib 'code' module was already imported, remove it so our package can load
if "code" in sys.modules:
    mod = sys.modules["code"]
    is_pkg = hasattr(mod, "__path__")
    points_to_ims = is_pkg and any(os.path.abspath(pp).startswith(os.path.abspath(IMS_CODE)) for pp in mod.__path__)
    if (not is_pkg) or (not points_to_ims):
        del sys.modules["code"]

# --- Create `obj` if missing (re-uses the stored repository document) ---
if "obj" not in globals():
    from hera.datalayer.project import Project

    PROJECT = globals().get("PROJECT", "UnitTestProject")
    proj = Project(projectName=PROJECT)
    docs = proj.getMeasurementsDocuments(
        type="ToolkitDataSource",
        repository="IMS",
        datasourceName="IMS",
    )
    assert docs, "ToolkitDataSource 'IMS' in repository 'IMS' not found. Run the register cell first."
    doc = docs[0]

    module_name, _, class_name = doc.desc["classpath"].rpartition(".")
    Cls = getattr(importlib.import_module(module_name), class_name)
    params = dict(doc.desc.get("parameters", {}))
    obj = Cls(**params)
    print("✅ Instantiated IMS object:", obj)

# --- Optional examples (disabled by default) ---
RUN_EXAMPLES = False          # set to True to actually run download/update
station_name = "YAVNEEL"      # change to a station that exists for you

if RUN_EXAMPLES:
    token_path = os.path.join(IMS_ROOT, "token.json")
    if not os.path.isfile(token_path):
        raise FileNotFoundError(f"Missing token file: {token_path}")

    # quick validation of token file
    try:
        with open(token_path, "r") as f:
            t = json.load(f)
        assert "Authorization" in t, "token.json must include an 'Authorization' field"
    except Exception as e:
        raise RuntimeError(f"Invalid token.json: {e}")

    print(f"▶ Running download/update for station '{station_name}' ...")
    try:
        obj.download(station=station_name, start="2020-01-01", end="latest")
        obj.update(station=station_name, end="latest")
        print("✔ Done")
    except Exception as e:
        print("⚠️ Download/update failed:", repr(e))
else:
    print("Examples are disabled (RUN_EXAMPLES = False). Set it to True to run download/update once your token is configured.")


qgl is not installed
Thingsboard interface not installed. Use pip install tb_rest_client.
📥 presenation.__init__ called
   📌 dataLayer = <class 'hera.measurements.meteorology.lowfreqdata.toolkit.lowFreqToolKit'>
   📌 analysis = <class 'hera.measurements.meteorology.lowfreqdata.analysis.analysis'>
📥 SeasonalPlots.__init__ called
📥 Plots.__init__ called
   📌 Received presentation object of type: <class 'hera.measurements.meteorology.lowfreqdata.presentationLayer.presenation'>
📥 DailyPlots.__init__ called
📥 Plots.__init__ called
   📌 Received presentation object of type: <class 'hera.measurements.meteorology.lowfreqdata.presentationLayer.presenation'>
✅ Instantiated IMS object: <code.IMS_experiment.IMS_experiment object at 0x7c94c41b94f0>
Examples are disabled (RUN_EXAMPLES = False). Set it to True to run download/update once your token is configured.


## 7) Troubleshooting Cheatsheet
- **ImportError: `presentation` not found** → Ensure `IMS_ROOT/code` and `IMS_ROOT` are both on `sys.path`. The class may import `presentation` as a sibling of `IMS_experiment.py`.
- **`argosDataObjects` / `pyargos` errors** → Verify `~/pyargos-master` exists and is on `PYTHONPATH`/`sys.path`.
- **CLI crashes on third-party warnings** → They are usually harmless (e.g., `pkg_resources` deprecation). Errors about `typing_extensions` often stem from a conflicting site-packages; ensure your venv site-packages precedes any system user-site.
- **Repository not found** → The CLI requires `--repository`. Make sure you pass the same name later when querying documents.
